In [1]:
import cobra

In [2]:
model = cobra.io.read_sbml_model("model.xml")

In [3]:
# ── 1. Existing metabolites (should already be in your model) ──────────────
pep_c = model.metabolites.get_by_id("cpd00061_c0")   # cytosolic PEP
pi_c  = model.metabolites.get_by_id("cpd00009_c0")   # cytosolic phosphate
pi_e  = model.metabolites.get_by_id("cpd00009_e0")   # extracellular phosphate

In [4]:
pep_c

Metabolite identifier,cpd00061_c0
Name,Phosphoenolpyruvate
Memory address,0x117ee1510
Formula,C3H2O6P
Compartment,c0
In 12 reaction(s),"rxn05226_c0, rxn00251_c0, rxn00247_c0, rxn00148_c0, rxn00745_c0, rxn00461_c0, rxn02476_c0, rxn01332_c0, rxn01316_c0, rxn00147_c0, rxn02331_c0, rxn00459_c0"


In [5]:
pi_c

Metabolite identifier,cpd00009_c0
Name,Phosphate
Memory address,0x117ef5a50
Formula,HO4P
Compartment,c0
In 145 reaction(s),"rxn00748_c0, rxn10265_c0, rxn02404_c0, rxn03409_c0, rxn01366_c0, rxn03407_c0, rxn01332_c0, rxn00132_c0, rxn08762_c0, rxn00781_c0, rxn03147_c0, rxn00187_c0, rxn09101_c0, rxn01255_c0, rxn05148_c0,..."


In [6]:
pi_e

Metabolite identifier,cpd00009_e0
Name,Phosphate [e0]
Memory address,0x117e40b90
Formula,HO4P
Compartment,e0
In 5 reaction(s),"rxn05145_c0, rxn05312_c0, rxn05313_c0, EX_cpd00009_e0, rxn08642_c0"


In [8]:
# Define and add an external version of PEP
pep_e = cobra.Metabolite(
    "cpd00061_e0",
    name="Phosphoenolpyruvate",
    formula="C3H2O6P",   # matches ModelSEED cpd00061 → keeps balance
    charge=-3,
    compartment="e0",
)
model.add_metabolites([pep_e])

In [10]:
model.metabolites.cpd00061_e0

Metabolite identifier,cpd00061_e0
Name,Phosphoenolpyruvate
Memory address,0x119d2c3d0
Formula,C3H2O6P
Compartment,e0
In 0 reaction(s),


In [11]:
# Define the antiporter
t = cobra.Reaction("rxn30513_c0")
t.name = "Phosphoenolpyruvate transport via phosphate antiport"
t.lower_bound = -1000.0
t.upper_bound = 1000.0
t.add_metabolites({
    pi_e:  -1.0,
    pep_c: -1.0,
    pi_c:   1.0,
    pep_e:  1.0,
})

In [12]:
# Add the transporter
model.add_reactions([t])

In [13]:
# Add an exchange reation for the external PEP
model.add_boundary(pep_e, type="exchange", lb=0.0, ub=1000.0)

Reaction identifier,EX_cpd00061_e0
Name,Phosphoenolpyruvate exchange
Memory address,0x119d7ffd0
Stoichiometry,cpd00061_e0 --> Phosphoenolpyruvate -->
GPR,
Lower bound,0.0
Upper bound,1000.0


In [14]:
# Write the model
cobra.io.write_sbml_model(model, "model.xml")